In [1]:
import os
import sys
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

from src.data.load_data import cargar_datos_tipo_data

# Agregar src/ al path si no está
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
# Importar funciones y cargar archivo csv
from src.data.load_data import cargar_datos_tipo_data

columnas_adult = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Ruta del archivo CSV
ruta_csv = "../data/raw/adult.data"

# Sw carga la data
df = cargar_datos_tipo_data(ruta_csv, columnas_adult)

2025-07-11 08:27:28,133 - INFO - Datos cargados correctamente desde: ../data/raw/adult.data


In [3]:
# 1. Imputar registros de valores perdidos
from src.preprocessing.preparacion import imputar_categoricas_por_arbol

# Se evalúa la imputación mediante métrica de f1-score:
# f1-score > 0.7 -> Imputación por árbol
# f1-score < 0.7 -> Imputación clase "Desconocido"

df = imputar_categoricas_por_arbol(df, variables=["workclass", "occupation", "native-country"], f1_umbral=0.7)

2025-07-11 08:27:30,806 - INFO - workclass - F1-score promedio: 0.674
2025-07-11 08:27:30,811 - WARNING - workclass: F1=0.674 bajo. Imputado como 'Desconocido'
2025-07-11 08:27:31,390 - INFO - occupation - F1-score promedio: 0.303
2025-07-11 08:27:31,394 - WARNING - occupation: F1=0.303 bajo. Imputado como 'Desconocido'
C:\Proyectos_Pycharm\Laboratorio_16_Aplicaciones_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
2025-07-11 08:27:32,125 - INFO - native-country - F1-score promedio: 0.891
2025-07-11 08:27:32,287 - INFO - native-country: Imputación completada con árbol (F1=0.891)


In [4]:
# Evaluar proporción de "Desconocidos"
for var in ["workclass", "occupation", "native-country"]:
    porcentaje = (df[var] == "Desconocido").mean()
    logging.info(f"{var}: {porcentaje:.2%} registros 'Desconocido'")

2025-07-11 08:27:32,331 - INFO - workclass: 5.64% registros 'Desconocido'
2025-07-11 08:27:32,335 - INFO - occupation: 5.66% registros 'Desconocido'
2025-07-11 08:27:32,339 - INFO - native-country: 0.00% registros 'Desconocido'


Se ha imputado con un árbol de decisión la variable "native-country" y se ha clasificado como categoría "Desconocido" en las variables "workclass" y "occupation" ya que el árbol no tuvo un buen desempeño.

In [5]:
# Convertir variables 'capital-gain' y 'capital-loss' a tipo binarias categoricas
df["capital-gain"] = (df["capital-gain"] > 0).astype("category")
df["capital-loss"] = (df["capital-loss"] > 0).astype("category")

logging.info("Variables 'capital-gain' y 'capital-loss' convertidas a binarias categóricas (True/False).")

2025-07-11 08:27:32,386 - INFO - Variables 'capital-gain' y 'capital-loss' convertidas a binarias categóricas (True/False).


In [6]:
# Separar variables numéricas y categóricas
from src.preprocessing.preparacion import separar_variables

df_num, df_cat = separar_variables(df)

2025-07-11 08:27:32,434 - INFO - Variables numéricas: ['age', 'fnlwgt', 'education-num', 'hours-per-week']
2025-07-11 08:27:32,434 - INFO - Variables categóricas: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'native-country', 'income']


In [7]:
# 2. Simetrización de los datos
from src.preprocessing.preparacion import detectar_asimetria, transformar_por_asimetria

df_num, resumen = transformar_por_asimetria(df_num, save_dir="../outputs/02_preparacion/01_simetrizacion")

2025-07-11 08:27:34,251 - INFO - age: gráfico guardado en ../outputs/02_preparacion/01_simetrizacion\age.png
2025-07-11 08:27:34,252 - INFO - age: sin transformación (ninguna mejora el skew, original = 0.56)
2025-07-11 08:27:35,400 - INFO - fnlwgt: gráfico guardado en ../outputs/02_preparacion/01_simetrizacion\fnlwgt.png
2025-07-11 08:27:35,401 - INFO - fnlwgt: sqrt aplicada (skew original = 1.45, final = 0.19)
2025-07-11 08:27:36,918 - INFO - education-num: gráfico guardado en ../outputs/02_preparacion/01_simetrizacion\education-num.png
2025-07-11 08:27:36,919 - INFO - education-num: sin transformación (simétrica, skew = -0.31)
2025-07-11 08:27:38,216 - INFO - hours-per-week: gráfico guardado en ../outputs/02_preparacion/01_simetrizacion\hours-per-week.png
2025-07-11 08:27:38,217 - INFO - hours-per-week: sin transformación (simétrica, skew = 0.23)
2025-07-11 08:27:38,226 - INFO - Resumen de transformaciones guardado en HTML: ../outputs/02_preparacion/01_simetrizacion\resumen_transform

La variable 'fnwlgt' tenía un índice de skew de 1.447, lo que indicaba una asimetría con cola derecha. Se le aplicó una transformación de raíz cuadrado, y ahora se tiene un índice de 0.189, lo cual indica una mayor simetría en la data.

In [8]:
# 3. Normalización de los datos
from src.preprocessing.preparacion import estandarizar_variables

df_normalizado = estandarizar_variables(
    df_num,
    save_dir="../outputs/02_preparacion/02_normalizacion"
)

2025-07-11 08:27:38,265 - INFO - Estandarización aplicada a variables numéricas
2025-07-11 08:27:38,814 - INFO - Gráfico guardado: ../outputs/02_preparacion/02_normalizacion\age_estandarizado.png
2025-07-11 08:27:39,300 - INFO - Gráfico guardado: ../outputs/02_preparacion/02_normalizacion\fnlwgt_estandarizado.png
2025-07-11 08:27:39,811 - INFO - Gráfico guardado: ../outputs/02_preparacion/02_normalizacion\education-num_estandarizado.png
2025-07-11 08:27:40,356 - INFO - Gráfico guardado: ../outputs/02_preparacion/02_normalizacion\hours-per-week_estandarizado.png


In [9]:
# 4. Detección de outliers
from src.preprocessing.preparacion import detectar_outliers_univariado

# Detección de outliers univariados
ruta_outliers_uni = "../outputs/02_preparacion/03_outliers"

df_outliers_uni = detectar_outliers_univariado(df_normalizado,
                                               ruta_salida=ruta_outliers_uni)

2025-07-11 08:27:40,485 - INFO - age: 166 outliers univariados
2025-07-11 08:27:40,489 - INFO - fnlwgt: 409 outliers univariados
2025-07-11 08:27:40,493 - INFO - education-num: 1198 outliers univariados
2025-07-11 08:27:40,498 - INFO - hours-per-week: 9008 outliers univariados


In [10]:
# Detección de outliers multivariados
from src.preprocessing.preparacion import detectar_outliers_mahalanobis

ruta_outliers_multi = "../outputs/02_preparacion/03_outliers"

outliers_maha = detectar_outliers_mahalanobis(
    df_normalizado,
    umbral=0.99,
    ruta_salida=ruta_outliers_multi
)

2025-07-11 08:27:40,683 - INFO - Outliers multivariados detectados: 989


Se observa que hay una distribución de las distancias de Mahalanobis con cola derecha, donde
bajo una distribución chi-cuadrado se tienen 989 valores atípicos fuera del umbral del 99%.

In [11]:
# Eliminar registros con outliers multivariados
n_outliers = outliers_maha.sum()
n_original = df_normalizado.shape[0]
df_sin_outliers = df_normalizado.loc[~outliers_maha]

logging.info(f"Outliers multivariados detectados: {n_outliers} ({round(n_outliers / n_original * 100, 2)}%) eliminados.")
logging.info(f"Filas finales después de eliminar outliers: {df_sin_outliers.shape[0]}")

2025-07-11 08:27:41,608 - INFO - Outliers multivariados detectados: 989 (3.04%) eliminados.
2025-07-11 08:27:41,609 - INFO - Filas finales después de eliminar outliers: 31572


In [12]:
# 5. Correlación

# Generar matriz de correlación
from src.preprocessing.preparacion import generar_matriz_correlacion

matriz = generar_matriz_correlacion(
    df_sin_outliers,
    save_path="../outputs/02_preparacion/04_correlacion/matriz_correlacion.png"
)

2025-07-11 08:27:41,942 - INFO - Matriz de correlación guardada en: ../outputs/02_preparacion/04_correlacion/matriz_correlacion.png


La matriz de correlación muestra que las correlaciones entre las variables se encuentran entre 0 y 0.17, por lo que se demuestra que no hay una colinealidad fuerte en las variables, generando redundancia estadística, por lo que no es necesario reducir la dimensionalidad.

In [13]:
# Homogenizar tamaño del dataframe numérico y categórico

# Filtrar variables categóricas según índices sin outliers
df_cat_filtrado = df_cat.loc[df_sin_outliers.index]

# Verificación de consistencia
if df_cat_filtrado.shape[0] == df_sin_outliers.shape[0]:
    logging.info(f"DataFrames alineados correctamente: {df_cat_filtrado.shape[0]} filas en ambos.")
else:
    logging.warning(f"Desalineación de filas: df_cat_filtrado={df_cat_filtrado.shape[0]}, df_sin_outliers={df_sin_outliers.shape[0]}")

2025-07-11 08:27:41,999 - INFO - DataFrames alineados correctamente: 31572 filas en ambos.


In [14]:
# Concatenar variables numéricas y categóricas para el modelado
df_modelo = pd.concat([df_sin_outliers, df_cat_filtrado], axis=1)

# Logging del resultado final
logging.info(f"Data final para modelado: {df_modelo.shape[0]} filas, {df_modelo.shape[1]} columnas")

# Verificar si hay nulos en df_modelo
nulos_modelo = df_modelo.isnull().sum()
nulos_totales = nulos_modelo.sum()

if nulos_totales == 0:
    logging.info("No se encontraron valores nulos en df_modelo.")
else:
    logging.warning(f"Se encontraron {nulos_totales} valores nulos en df_modelo:")
    logging.warning("\n" + str(nulos_modelo[nulos_modelo > 0]))

2025-07-11 08:27:42,067 - INFO - Data final para modelado: 31572 filas, 15 columnas
2025-07-11 08:27:42,085 - INFO - No se encontraron valores nulos en df_modelo.


In [15]:
# Binarizar la variable objetivo (1 si >50K, 0 si <=50K)
df_modelo["income"] = (df_modelo["income"] == ">50K").astype(int)

# Verificación y logging
valores_unicos = df_modelo["income"].unique()
logging.info(f"Variable 'income' binarizada. Valores únicos: {valores_unicos.tolist()}")

2025-07-11 08:27:42,156 - INFO - Variable 'income' binarizada. Valores únicos: [0, 1]


In [16]:
# 6. Information Value

from src.preprocessing.preparacion import calcular_information_value

# Calcular el IV de todo el dataset
iv_total = calcular_information_value(
    df=df_modelo,
    target="income",
    save_path="../outputs/02_preparacion/05_information_value/iv_completo.html"
)

2025-07-11 08:27:42,615 - INFO - Information Value exportado a HTML: ../outputs/02_preparacion/05_information_value/iv_completo.html
2025-07-11 08:27:42,616 - INFO - Cálculo de Information Value completado.


In [17]:
# Eliminar variables con IV < 0.3 (Filtrar predictores fuertes)
from src.preprocessing.preparacion import eliminar_variables
iv_variables = ["workclass", "capital-loss", "native-country", "race", "fnlwgt"]
df_modelo = eliminar_variables(df_modelo, iv_variables)

2025-07-11 08:27:42,670 - INFO - Variables eliminadas: ['workclass', 'capital-loss', 'native-country', 'race', 'fnlwgt']


In [18]:
# 7. Dumificación de variables categóricas
from src.preprocessing.preparacion import dumificar_variables

df_modelo = dumificar_variables(
    df=df_modelo,
    target="income",
    save_path="../outputs/02_preparacion/06_dummizacion/resumen_dummies.html"
)

2025-07-11 08:27:42,761 - INFO - Columnas convertidas a 'category': ['education', 'marital-status', 'occupation', 'relationship', 'sex']
2025-07-11 08:27:42,773 - INFO - Dumificación aplicada. Variables categóricas transformadas: 6
2025-07-11 08:27:42,779 - INFO - Resumen de columnas dummificadas guardado en: ../outputs/02_preparacion/06_dummizacion/resumen_dummies.html


In [20]:
from pathlib import Path

# Crear carpeta si no existe
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Exportar a CSV
ruta_csv = "../data/processed/adult_limpio.csv"
df_modelo.to_csv(ruta_csv, index=False)

# Log final
logging.info(f"Data limpia exportada a {ruta_csv}")

2025-07-11 08:28:26,546 - INFO - Data limpia exportada a ../data/processed/adult_limpio.csv


In [21]:
df_modelo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31572 entries, 0 to 32560
Data columns (total 46 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   age                                   31572 non-null  float64
 1   education-num                         31572 non-null  float64
 2   hours-per-week                        31572 non-null  float64
 3   income                                31572 non-null  int64  
 4   education_11th                        31572 non-null  bool   
 5   education_12th                        31572 non-null  bool   
 6   education_1st-4th                     31572 non-null  bool   
 7   education_5th-6th                     31572 non-null  bool   
 8   education_7th-8th                     31572 non-null  bool   
 9   education_9th                         31572 non-null  bool   
 10  education_Assoc-acdm                  31572 non-null  bool   
 11  education_Assoc-voc 